# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [35]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE


# Standard seed used throughout the notebook, for reproducibility
SEED = 1

In [7]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

### **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?


Let's check it. First let's see the kind of data that we have, then how many fraud/non fraud distribution we have.

In [9]:
fraud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   distance_from_home              1000000 non-null  float64
 1   distance_from_last_transaction  1000000 non-null  float64
 2   ratio_to_median_purchase_price  1000000 non-null  float64
 3   repeat_retailer                 1000000 non-null  float64
 4   used_chip                       1000000 non-null  float64
 5   used_pin_number                 1000000 non-null  float64
 6   online_order                    1000000 non-null  float64
 7   fraud                           1000000 non-null  float64
dtypes: float64(8)
memory usage: 61.0 MB


We have not nulls, which is good.

In [10]:
fraud.describe()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,26.628792,5.036519,1.824182,0.881536,0.350399,0.100608,0.650552,0.087403
std,65.390784,25.843093,2.799589,0.323157,0.477095,0.300809,0.476796,0.282425
min,0.004874,0.000118,0.004399,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.878008,0.296671,0.475673,1.000000,0.000000,0.000000,0.000000,0.000000
50%,9.967760,0.998650,0.997717,1.000000,0.000000,0.000000,1.000000,0.000000
75%,25.743985,3.355748,2.096370,1.000000,1.000000,0.000000,1.000000,0.000000
max,10632.723672,11851.104565,267.802942,1.000000,1.000000,1.000000,1.000000,1.000000


In [14]:
print(fraud["fraud"].value_counts())

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64


It's very imbalanced, we have 91% of non fraud and a 9% for fraud.

### **2.** Train a LogisticRegression.


In [18]:
#Gather the names of the columns in a list so it's easier to copy-paste them into the code below
fraud.columns.tolist()

['distance_from_home',
 'distance_from_last_transaction',
 'ratio_to_median_purchase_price',
 'repeat_retailer',
 'used_chip',
 'used_pin_number',
 'online_order',
 'fraud']

In [21]:
#Step 1: Split the data into training and testing sets. Use a 80/20 split, and set the random_state to SEED.
# Select the features (X) and the target (y).

X = fraud[['distance_from_home',
 'distance_from_last_transaction',
 'ratio_to_median_purchase_price',
 'repeat_retailer',
 'used_chip',
 'used_pin_number',
 'online_order']]
y = fraud['fraud']

In [23]:
# Step 2: Train / Test Split
# ---------------------------
# Train / Test Split (without stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

# Let's look at the fraud (1) rate:
print('Fraud proportion in y_train:')
print(y_train.value_counts(normalize=True))
print()
print('Fraud proportion in y_test:')
print(y_test.value_counts(normalize=True))

X_train shape: (800000, 7)
X_test shape:  (200000, 7)
Fraud proportion in y_train:
fraud
0.0    0.912597
1.0    0.087402
Name: proportion, dtype: float64

Fraud proportion in y_test:
fraud
0.0    0.912595
1.0    0.087405
Name: proportion, dtype: float64


In [26]:
# Step 3 - Feature Engineering: Feature Scaling
# ------------------------------------------------

# SGDRegressor --> standardize with StandardScaler --> scaling parameters: mean and std
scaler = StandardScaler()
scaler.set_output(transform="pandas")
# set_output(transform="pandas") makes the scaler return a DataFrame (keeps the real column names and index)

# Fit ONLY on X_train --> avoids leaking any information from the test set into the scaling parameters
X_train_scaled_df = scaler.fit_transform(X_train)

# Then apply that same transformation to X_test
X_test_scaled_df = scaler.transform(X_test)

# BEFORE: original scale
print('BEFORE scaling (X_train):')
print(X_train.describe().loc[['mean', 'std']].round(4))
print()

# AFTER: standardized scale
print('AFTER scaling (X_train):')
print(X_train_scaled_df.describe().loc[['mean', 'std']].round(4))

BEFORE scaling (X_train):
      distance_from_home  distance_from_last_transaction  \
mean             26.7017                          5.0398   
std              66.5358                         26.3318   

      ratio_to_median_purchase_price  repeat_retailer  used_chip  \
mean                          1.8218           0.8816     0.3504   
std                           2.8080           0.3230     0.4771   

      used_pin_number  online_order  
mean           0.1004        0.6508  
std            0.3006        0.4767  

AFTER scaling (X_train):
      distance_from_home  distance_from_last_transaction  \
mean                 0.0                             0.0   
std                  1.0                             1.0   

      ratio_to_median_purchase_price  repeat_retailer  used_chip  \
mean                            -0.0              0.0        0.0   
std                              1.0              1.0        1.0   

      used_pin_number  online_order  
mean              0.0   

In [37]:
# Step 4 - Model Training: SGDClassifier
# --------------------------------------

# loss='log_loss' --> same Sigmoid + Cost Function from the theory!
baseline_model = SGDClassifier(loss='log_loss', random_state=SEED)
baseline_model.fit(X_train_scaled_df, y_train)  # Fit on the original, imbalanced Train set

print("Model trained!")
print()

# The learned weights (w) = one per feature
weights = pd.Series(baseline_model.coef_[0], index=X_train_scaled_df.columns)
# In Regression the answer is a single continuous value --> one coefficient vector --> coef_
# In binary Classification the answer is also a single outcome --> one coefficient vector --> coef_[0]
# (already distinguishes between the two classes) 
# In multiclass Classification, each class needs its own weight vectors --> coef_[0], coef_[1], coef_[2]...

# Sorted by |w| so it is easy to read, but keeping the sign to show the direction of the effect
print('Learned weights (Wi):')
print(weights.round(4).reindex(abs(weights).sort_values(ascending=False).index))

Model trained!

Learned weights (Wi):
used_pin_number                  -3.4432
online_order                      2.9391
ratio_to_median_purchase_price    2.4989
distance_from_home                1.0072
distance_from_last_transaction    0.5461
used_chip                        -0.5085
repeat_retailer                  -0.2267
dtype: float64


In [38]:
# Step 5 - Results
# ----------------

# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = baseline_model.predict(X_train_scaled_df)     # train
y_pred_test = baseline_model.predict(X_test_scaled_df)       # test

accuracy_train = accuracy_score(y_train, y_pred_train)  # train, just to compare with the report below
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

# Classification Report (Test)
# ----------------------------
print(classification_report(y_test, y_pred_test, target_names=['non fraud', 'fraud']))

Accuracy (Train): 95.94%

              precision    recall  f1-score   support

   non fraud       0.96      0.99      0.98    182519
       fraud       0.89      0.61      0.72     17481

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000



### **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.


The SGD classifier achieved an overall training accuracy of 95.94%. The model performed very well in identifying non-fraudulent transactions, with a precision of 0.96 and a recall of 0.99. For fraudulent transactions, the precision was 0.89, indicating that most transactions predicted as fraudulent were indeed fraud. However, the recall was 0.61, meaning that the model detected only 61% of all fraudulent transactions while missing approximately 39%. Since fraud detection typically prioritizes identifying as many fraudulent cases as possible, this relatively low recall represents the main limitation of the model, despite its high overall accuracy.

### **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 

In [39]:
# Step 4 - Model Training: SGDClassifier (Oversampling)
# ---------------------------------------------------------

# RandomOverSampler --> duplicates random instances from the minority class (Fraud)
# until it matches the size of the majority class (No Fraud)
ros = RandomOverSampler(random_state=SEED)
X_train_over, y_train_over = ros.fit_resample(X_train_scaled_df, y_train)

print('Before:', y_train.value_counts())
print('After: ', y_train_over.value_counts())

oversampling_model = SGDClassifier(loss='log_loss', random_state=SEED)
oversampling_model.fit(X_train_over, y_train_over)  # Fit on the oversampled Train set

print("Model trained!")

Before: fraud
0.0    730078
1.0     69922
Name: count, dtype: int64
After:  fraud
1.0    730078
0.0    730078
Name: count, dtype: int64
Model trained!


In [41]:
# Step 5 - Results
# ----------------

# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = oversampling_model.predict(X_train_over)     # train (oversampled)
y_pred_test = oversampling_model.predict(X_test_scaled_df)  # test (untouched!)

accuracy_train = accuracy_score(y_train_over, y_pred_train)
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

# Classification Report (Test)
# ----------------------------
print(classification_report(y_test, y_pred_test, target_names=['No Fraud', 'Fraud'], zero_division=0))

Accuracy (Train): 93.85%

              precision    recall  f1-score   support

    No Fraud       0.99      0.93      0.96    182519
       Fraud       0.57      0.95      0.71     17481

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



After applying oversampling to address the class imbalance, the model's overall accuracy decreased from 95.94% to 93.85%. However, the recall for the fraud class increased substantially from 0.61 to 0.95, indicating that the model was able to identify almost all fraudulent transactions. This improvement came at the expense of precision, which dropped from 0.89 to 0.57, meaning that more legitimate transactions were incorrectly classified as fraudulent. Such a trade-off is expected when using oversampling techniques and may be desirable in fraud detection, where failing to detect fraudulent activity is often more costly than investigating additional false alarms.

### **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?

In [42]:
# Step 4 - Model Training: SGDClassifier (Undersampling)
# ------------------------------------------------------

# RandomUnderSampler --> removes random instances from the majority class (No Fraud)
# until it matches the size of the minority class (Fraud)
rus = RandomUnderSampler(random_state=SEED)
X_train_under, y_train_under = rus.fit_resample(X_train_scaled_df, y_train)

print('Before:', y_train.value_counts())
print('After: ', y_train_under.value_counts())

undersampling_model = SGDClassifier(loss='log_loss', random_state=SEED)
undersampling_model.fit(X_train_under, y_train_under)  # Fit on the undersampled Train set

print("Model trained!")

Before: fraud
0.0    730078
1.0     69922
Name: count, dtype: int64
After:  fraud
0.0    69922
1.0    69922
Name: count, dtype: int64
Model trained!


In [43]:
# Step 5 - Results
# ----------------

# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = undersampling_model.predict(X_train_under)   # train (undersampled)
y_pred_test = undersampling_model.predict(X_test_scaled_df) # test (untouched!)

accuracy_train = accuracy_score(y_train_under, y_pred_train)
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

# Classification Report (Test)
# ----------------------------
print(classification_report(y_test, y_pred_test, target_names=['No Fraud', 'Fraud'], zero_division=0))

Accuracy (Train): 94.28%

              precision    recall  f1-score   support

    No Fraud       1.00      0.92      0.96    182519
       Fraud       0.55      0.96      0.70     17481

    accuracy                           0.93    200000
   macro avg       0.77      0.94      0.83    200000
weighted avg       0.96      0.93      0.94    200000



After applying undersampling, the SGD classifier achieved a training accuracy of 94.28%. The model showed a substantial improvement in detecting fraudulent transactions, with the recall for the fraud class increasing to 0.96, meaning that almost all fraud cases were correctly identified. However, this improvement came at the expense of precision, which decreased to 0.55, indicating that a larger number of legitimate transactions were incorrectly classified as fraudulent. Overall, undersampling shifted the model towards maximizing fraud detection while accepting a higher rate of false positives.

### **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [44]:
# Step 4 - Model Training: SGDClassifier (SMOTE)
# ----------------------------------------------

# SMOTE --> generates synthetic samples for the minority class (Fraud) via KNN interpolation,
# instead of duplicating existing points, until it matches the size of the majority class
sm = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = sm.fit_resample(X_train_scaled_df, y_train)

print('Before:', y_train.value_counts())
print('After: ', y_train_smote.value_counts())

smote_model = SGDClassifier(loss='log_loss', random_state=SEED)
smote_model.fit(X_train_smote, y_train_smote)  # Fit on the SMOTE-resampled Train set

print("Model trained!")

Before: fraud
0.0    730078
1.0     69922
Name: count, dtype: int64
After:  fraud
1.0    730078
0.0    730078
Name: count, dtype: int64
Model trained!


In [45]:
# Step 5 - Results
# ----------------

y_pred_train = smote_model.predict(X_train_smote)
y_pred_test = smote_model.predict(X_test_scaled_df)

accuracy_train = accuracy_score(y_train_smote, y_pred_train)
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

print(classification_report(y_test, y_pred_test, target_names=['No Fraud', 'Fraud'], zero_division=0))

Accuracy (Train): 93.88%

              precision    recall  f1-score   support

    No Fraud       0.99      0.93      0.96    182519
       Fraud       0.57      0.95      0.71     17481

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



After applying SMOTE (Synthetic Minority Over-sampling Technique), the SGD classifier achieved a training accuracy of 93.88%. The model significantly improved its ability to detect fraudulent transactions, achieving a recall of 0.95 for the fraud class, indicating that most fraud cases were correctly identified. However, this improvement came at the expense of precision, which decreased to 0.57, meaning that more legitimate transactions were incorrectly classified as fraudulent. Overall, SMOTE effectively increased the model's sensitivity to fraud while sacrificing precision, resulting in a trade-off between detecting fraudulent transactions and minimizing false positives.

### Conclusion

Although the original SGD classifier achieved the highest overall accuracy (95.94%), it detected only 61% of fraudulent transactions. The sampling techniques substantially improved fraud detection, with recall increasing to approximately 95–96%. Among these methods, SMOTE provided the best overall balance, achieving a high fraud recall (0.95) while maintaining slightly better precision than undersampling. Therefore, SMOTE would be the preferred model for fraud detection, where correctly identifying fraudulent transactions is typically more important than maximizing overall accuracy.